In [ ]:
%pip install openai scikit-learn pandas

In [ ]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [ ]:
SYSTEM_PROMPT = """
You are a HAM-D (Hamilton Depression Rating Scale, 21-item) survey scoring assistant.
Your job is to extract answers for the HAM-D-21 based on a given conversation transcript.
The HAM-D-21 is a clinician-administered assessment that measures the severity of depression over the past week.

Each item below lists its description and its allowed score range. Different items use different scales:

q1  Depressed mood (sadness, hopelessness, helplessness, worthlessness)        scale 0-4
q2  Feelings of guilt (self-reproach, ideas of guilt, delusions of guilt)       scale 0-4
q3  Suicide (feels life not worth living, wishes dead, ideation, attempts)      scale 0-4
q4  Insomnia early (difficulty falling asleep)                                  scale 0-2
q5  Insomnia middle (waking during the night, restless sleep)                   scale 0-2
q6  Insomnia late (waking in early hours and unable to fall back asleep)        scale 0-2
q7  Work and activities (loss of interest, fatigue, decreased productivity)     scale 0-4
q8  Retardation (slowness of thought, speech, movement) - usually observed      scale 0-4
q9  Agitation (fidgeting, hand-wringing, hair-pulling) - usually observed       scale 0-4
q10 Anxiety - psychic (subjective tension, worry, irritability, fears)          scale 0-4
q11 Anxiety - somatic (GI, cardiovascular, respiratory, sweating, headaches)    scale 0-4
q12 Somatic symptoms - gastrointestinal (loss of appetite, heavy stomach)       scale 0-2
q13 Somatic symptoms - general (heaviness, fatigue, backache, muscle aches)     scale 0-2
q14 Genital symptoms (loss of libido, menstrual disturbances)                   scale 0-2
q15 Hypochondriasis (preoccupation with bodily health, conviction of illness)   scale 0-4
q16 Loss of weight (by history or measured)                                     scale 0-2
q17 Insight (recognition of being depressed/ill)                                scale 0-2
q18 Diurnal variation (whether mood is worse at a particular time of day)       scale 0-2
q19 Depersonalization and derealization (feelings of unreality, dreamlike)      scale 0-4
q20 Paranoid symptoms (suspiciousness, ideas of reference, persecution)         scale 0-4
q21 Obsessional and compulsive symptoms (intrusive thoughts, rituals)           scale 0-2

General severity guide for each scale:
0 = absent / none
1 = mild / occasional / doubtful
2 = moderate / definite
3 = severe (only for 0-4 items)
4 = very severe / incapacitating (only for 0-4 items)

Specific scoring guidance for tricky items:
- q1: 0=absent, 1=indicated only on questioning, 2=spontaneously reported verbally, 3=communicated nonverbally (facial expression, voice, tearfulness), 4=patient reports virtually only these feelings.
- q3: 0=absent, 1=feels life not worth living, 2=wishes were dead, 3=suicidal ideas or gestures, 4=attempts at suicide.
- q4/q5/q6: 0=no difficulty, 1=occasional/mild, 2=nightly/severe.
- q7: 0=no difficulty, 1=thoughts of incapacity/fatigue, 2=loss of interest in activities, 3=decreased productivity, 4=stopped working due to illness.
- q8 and q9 are clinician-observed; if not evident from the transcript, score 0.
- q15: 0=not present, 1=self-absorption (bodily), 2=preoccupation with health, 3=frequent complaints/requests for help, 4=hypochondriacal delusions.
- q16: 0=no weight loss, 1=probable weight loss associated with illness, 2=definite weight loss.
- q17: 0=acknowledges being depressed and ill, 1=acknowledges illness but attributes cause to bad food/climate/overwork/virus/need for rest, 2=denies being ill at all.
- q18: 0=no variation, 1=mild variation worse in AM or PM, 2=severe variation.
- q20: 0=none, 1=suspicious, 2=ideas of reference, 3=delusions of reference and persecution, 4=hallucinations, persecutory.

Topic order in the conversation (use this as a guide; the assistant typically asks about these in roughly this order):
mood -> crying -> guilt / self-blame -> punishment -> suicide -> sleep onset (q4) -> sleep middle (q5) -> sleep early morning (q6) -> work / activities (q7) -> anxiety / worry (q10) -> physical anxiety symptoms (q11) -> appetite (q12, infer q16) -> energy / heaviness / aches (q13) -> sex (q14) -> health concerns (q15) -> diurnal variation (q18) -> unreal / dreamlike (q19) -> paranoid (q20) -> obsessive / compulsive (q21) -> insight (q17)

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value within that item's allowed range ("0"-"4" for 0-4 items, "0"-"2" for 0-2 items).
Rules:
- Match each topic in the conversation to the correct q number based on its description above (not by response position).
- Score each item based solely on what the user said in the transcript.
- For q8 and q9, score 0 unless the user clearly describes psychomotor retardation or agitation.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q21 key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "q8": "0",
  "q9": "0",
  "q10": "0",
  "q11": "0",
  "q12": "0",
  "q13": "0",
  "q14": "0",
  "q15": "0",
  "q16": "0",
  "q17": "0",
  "q18": "0",
  "q19": "0",
  "q20": "0",
  "q21": "0"
}
"""
MODEL = 'qwen3:8b'

- Use a json dataset of HAM-D-21 conversation transcripts
- Extract scores using LLM
- Validate whether they were correct using MSE

In [ ]:
# Load the HAM-D-21 dataset
import json

with open('hamd_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

In [ ]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

In [ ]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,  # low temperature for consistent scoring
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [ ]:
# Run predictions on training and test set
def generate_scores(conversations):
    scores = []
    for i, c in enumerate(conversations):
        print(f'Generating {i+1}/{len(conversations)}')
        scores.append(create_prediction(c))
    return scores

train_scores_raw = generate_scores(conv_train)
test_scores_raw = generate_scores(_conv_test)

In [ ]:
# Parse json scores
def parse_scores(raw_scores):
    parsed = []
    for s in scores:
    try:
        json_scores.append(json.loads(s))
    except json.JSONDecodeError:
        print(f'Failed to parse: {s}')
        json_scores.append({f'q{i}': '0' for i in range(1, 22)})
    return parsed

train_scores_parsed = parse_scores(train_scores_raw)
test_scores_parsed = parse_scores(test_scores_raw)

In [ ]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

def evaluate_split(parsed_scores, expected_scores, name):
    pred_df = pd.DataFrame(parsed_scores)
    expected_df = pd.DataFrame(expected_scores)

    cols = [f'q{i}' for i in range(1, 22)]
    pred_df = pred_df[cols]
    expected_df = expected_df[cols]

    mse = mean_squared_error(
        convert_scores_to_array(expected_df),
        convert_scores_to_array(pred_df)
    )
    print(f'{name} MSE: {mse}')

    display(expected_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(
        columns={'self': 'Expected', 'other': 'Predicted'}, level=1
    ))

evaluate_split(train_scores_parsed, score_train, 'Train')
evaluate_split(test_scores_parsed, _score_test, 'Test')